# SIXray-D: teacher and student inference

Comparing:
- baseline teacher (RT-DETRv2)
- oversampling teacher (RT-DETRv2)
- ResNet18 student

on the same validation split. 


In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / 'students/anna_student_resnet18/src')]

import matplotlib.pyplot as plt
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm


## Paths and settings


In [2]:
ARTIFACT_ROOT = Path(r'C:\Users\89526\Documents\sixray-artifacts')
PROJECT_DIR = ARTIFACT_ROOT / 'DatasetAPAI/SIXray_Project'
DATA_DIR = ARTIFACT_ROOT / 'DatasetAPAI/subset_clean'
ARCHIVE_PATH = ARTIFACT_ROOT / 'DatasetAPAI/subset_clean.zip'
SPLIT_PATH = PROJECT_DIR / 'splits/split_seed42_train10500_val300pos1200neg.json'
CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints_angelo'
STUDENT_PATH = PROJECT_DIR / 'student_checkpoints_anna/student_resnet18_yolo2_local_offsets_best_map_50.pth'

N_IMAGES = None #None evaluates the full validation split. Set int to evaluate first N images
SCORE_THRESHOLD = 0.001
DISPLAY_THRESHOLD = 0.25
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Validation data

Load the existing split.


In [3]:
from src.data.dataset import SixRayDataset
from src.engine.inference import prepare_validation_files, predict_image, plot_comparison
from sixray_student.data import SixRayStudentDataset, load_split

_, val_indices, _ = load_split(SPLIT_PATH)
val_indices = val_indices[:N_IMAGES]
prepare_validation_files(ARCHIVE_PATH, DATA_DIR, val_indices)


Loaded split: C:\Users\89526\Documents\sixray-artifacts\DatasetAPAI\SIXray_Project\splits\split_seed42_train10500_val300pos1200neg.json
Train: 10500
Val: 1500
Test: 13246


## Models and preprocessing

Load saved weights strictly, without downloading ImageNet weights. Teachers use their saved 640 x 640 processor with rescaling to [0,1]. The student loader returns resized [0,1] tensors; ImageNet normalization is inside the student. Architectures and decoding are unchanged.


In [4]:
from src.utils.checkpoint import load_checkpoint
from sixray_student.config import CLASS_NAMES
from sixray_student.model import build_student_model
from sixray_student.train_utils import load_checkpoint_for_training

teachers = {}
for name, folder in {
    'teacher_baseline': '01_rtdetr_teacher_baseline_angelo_best',
    'teacher_oversampling': '02_rtdetr_teacher_oversampling_angelo_best',
}.items():
    processor, model, _ = load_checkpoint(CHECKPOINT_DIR / folder, device=DEVICE, strict=True)
    teachers[name] = (processor, model.eval())

checkpoint = load_checkpoint_for_training(STUDENT_PATH, device=torch.device('cpu'))
student = build_student_model(pretrained_backbone=False)
student.load_state_dict(checkpoint['model_state_dict'], strict=True)
student.to(DEVICE).eval();


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

In [ ]:
processor = teachers['teacher_baseline'][0]
teacher_data = SixRayDataset(DATA_DIR / 'train/images', DATA_DIR / 'train.json', processor)
student_data = SixRayStudentDataset(DATA_DIR / 'train/images', DATA_DIR / 'train.json', val_indices)

# Both teachers share preprocessing; all models share the dataset's class IDs.
assert all(p.to_dict() == processor.to_dict() for p, _ in teachers.values())
assert checkpoint['class_names'] == CLASS_NAMES
assert {int(k): v for k, v in teacher_data.id2label.items()} == dict(enumerate(CLASS_NAMES))
assert all(m.config.id2label == dict(enumerate(CLASS_NAMES)) for _, m in teachers.values())


Dataset: C:\Users\89526\Documents\sixray-artifacts\DatasetAPAI\subset_clean\train.json
Images folder: C:\Users\89526\Documents\sixray-artifacts\DatasetAPAI\subset_clean\train\images
Images used here: 1500


## Evaluation

Evaluate all three models on the validation images.
- AP, AP50, AP75. 

All boxes use original-image pixel coordinates, including size-specific AP.


In [ ]:
metrics = {
    name: MeanAveragePrecision(
        box_format='xyxy', iou_type='bbox', class_metrics=True,
        max_detection_thresholds=[1, 10, 100],
    )
    for name in [*teachers, 'student']
}
examples = []
with torch.inference_mode():
    for position in tqdm(range(len(student_data)), desc='Validation'):
        sample = predict_image(position, teacher_data, student_data, teachers, student,
                               DEVICE, confidence=SCORE_THRESHOLD)
        for name, prediction in sample['predictions'].items():
            metrics[name].update([prediction], [sample['ground_truth']])
        if position < 5:
            examples.append(sample)

results = {name: {key: value.tolist() for key, value in metric.compute().items()}
           for name, metric in metrics.items()}

for name, result in results.items():
    print(f"{name}: AP={result['map']:.4f}, AP50={result['map_50']:.4f}, AP75={result['map_75']:.4f}")


Validation:   0%|          | 0/1500 [00:00<?, ?it/s]

teacher_baseline: AP=0.7003, AP50=0.8330, AP75=0.7386
teacher_oversampling: AP=0.7449, AP50=0.8662, AP75=0.7920
student: AP=0.3556, AP50=0.6797, AP75=0.3204


## Qualitative examples

Display ground truth and the three models side by side on the original RGB images, using consistent class colors.


In [ ]:
for sample in examples:
    fig = plot_comparison(sample, confidence=DISPLAY_THRESHOLD)
    plt.show()
    plt.close(fig)


## Save results



In [ ]:
result_dir = ARTIFACT_ROOT / 'evaluation_results' / (
    'validation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
)
result_dir.mkdir(parents=True, exist_ok=False)
summary = {
    'images_evaluated': len(student_data), 'split_path': str(SPLIT_PATH),
    'checkpoint_dir': str(CHECKPOINT_DIR), 'student_checkpoint': str(STUDENT_PATH),
    'score_threshold': SCORE_THRESHOLD, 'student_nms_iou': 0.5,
    'max_detections': 100, 'coco_max_dets': [1, 10, 100],
    'coordinate_frame': 'original_image_absolute_xyxy', 'metrics': results,
}
with (result_dir / 'results.json').open('x', encoding='utf-8') as output:
    json.dump(summary, output, indent=2, allow_nan=False)
print(result_dir / 'results.json')
